In [ ]:
# Google Colab에서 Car Damage Detection with YOLOv8m
# GPU 런타임 사용 권장: 런타임 > 런타임 유형 변경 > T4 GPU

# ==========================================
# 1. 환경 설정 및 패키지 설치
# ==========================================

!pip install ultralytics -q
!pip install roboflow -q

import os
import cv2
import numpy as np
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import yaml
import shutil
import json
from sklearn.model_selection import train_test_split
from google.colab import drive, files
from IPython.display import display, Image as IPImage
import zipfile
from collections import defaultdict

# GPU 확인
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 31.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Tue Jan 27 01:57:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-------

In [ ]:
# ==========================================
# 2. Google Drive 마운트 (선택사항)
# ==========================================

# 데이터셋이 구글 드라이브에 있는 경우
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
YOLOv8 차량 부위 분류 (32개 클래스)
학습, 검증, 추론을 하나의 파일로 통합
"""

from ultralytics import YOLO
import cv2
import numpy as np
import torch
import os
import yaml


class CarPartsYOLO:
    """차량 부위 감지 및 학습을 위한 통합 클래스"""

    def __init__(self):
        self.dataset_path = r"/content/drive/MyDrive/balanced_dataset_split"
        self.model = None
        self.class_names = None

        # 32개 차량 부위 클래스
        self.classes = {
            0: 'Front fender(L)', 1: 'Rear bumper', 2: 'Front Wheel(R)',
            3: 'Trunk lid', 4: 'Rocker panel(L)', 5: 'Front fender(R)',
            6: 'Front bumper', 7: 'Bonnet', 8: 'Rear Wheel(R)',
            9: 'Rear door(R)', 10: 'Front door(R)', 11: 'Head lights(R)',
            12: 'Rear fender(R)', 13: 'Rear fender(L)', 14: 'Rocker panel(R)',
            15: 'Rear lamp(L)', 16: 'Side mirror(R)', 17: 'Rear Wheel(L)',
            18: 'Rear door(L)', 19: 'Side mirror(L)', 20: 'Head lights(L)',
            21: 'Front Wheel(L)', 22: 'Front door(L)', 23: 'Rear lamp(R)',
            24: 'Windshield', 25: 'Roof', 26: 'Undercarriage',
            27: 'Rear windshield', 28: 'C pillar(L)', 29: 'A pillar(L)',
            30: 'C pillar(R)', 31: 'A pillar(R)'
        }

        # 부위별 색상 매핑
        self.colors = {
            'Front fender(L)': (0, 255, 0), 'Front fender(R)': (0, 255, 0),
            'Rear bumper': (255, 0, 0), 'Front bumper': (255, 0, 0),
            'Front Wheel(R)': (255, 255, 0), 'Front Wheel(L)': (255, 255, 0),
            'Rear Wheel(R)': (255, 255, 0), 'Rear Wheel(L)': (255, 255, 0),
            'Trunk lid': (0, 255, 255), 'Bonnet': (0, 255, 255),
            'Rocker panel(L)': (255, 0, 255), 'Rocker panel(R)': (255, 0, 255),
            'Rear door(R)': (0, 128, 255), 'Rear door(L)': (0, 128, 255),
            'Front door(R)': (0, 128, 255), 'Front door(L)': (0, 128, 255),
            'Head lights(R)': (255, 128, 0), 'Head lights(L)': (255, 128, 0),
            'Rear fender(R)': (128, 0, 255), 'Rear fender(L)': (128, 0, 255),
            'Rear lamp(L)': (0, 255, 128), 'Rear lamp(R)': (0, 255, 128),
            'Side mirror(R)': (128, 255, 0), 'Side mirror(L)': (128, 255, 0),
            'Windshield': (255, 128, 128), 'Rear windshield': (255, 128, 128),
            'Roof': (128, 128, 255), 'Undercarriage': (128, 255, 128),
            'C pillar(L)': (255, 255, 128), 'C pillar(R)': (255, 255, 128),
            'A pillar(L)': (128, 255, 255), 'A pillar(R)': (128, 255, 255)
        }

    def create_yaml_config(self):
        """데이터셋 YAML 설정 파일 생성"""
        # 절대 경로로 변환
        train_path = os.path.join(self.dataset_path, 'train', 'images')
        val_path = os.path.join(self.dataset_path, 'val', 'images')
        test_path = os.path.join(self.dataset_path, 'test', 'images')

        # 경로 존재 확인
        if not os.path.exists(train_path):
            raise FileNotFoundError(f"학습 이미지 경로를 찾을 수 없습니다: {train_path}")
        if not os.path.exists(val_path):
            raise FileNotFoundError(f"검증 이미지 경로를 찾을 수 없습니다: {val_path}")

        print(f"✓ 학습 경로 확인: {train_path}")
        print(f"✓ 검증 경로 확인: {val_path}")

        # YAML 설정 (절대 경로 사용)
        config = {
            'train': train_path,
            'val': val_path,
            'test': test_path if os.path.exists(test_path) else val_path,
            'nc': 32,
            'names': self.classes
        }

        yaml_path = 'car_parts.yaml'
        with open(yaml_path, 'w', encoding='utf-8') as f:
            yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

        print(f"✓ YAML 설정 파일 생성: {yaml_path}")
        return yaml_path

    def train(self,
              model_size='n',
              epochs=100,
              imgsz=640,
              batch=16,
              patience=20,
              lr0=0.01):
        """
        모델 학습

        Args:
            model_size: 모델 크기 (n, s, m, l, x)
            epochs: 학습 에포크 수
            imgsz: 이미지 크기
            batch: 배치 크기
            patience: Early stopping patience
            lr0: 초기 학습률
        """
        # YAML 파일 생성
        yaml_path = self.create_yaml_config()

        # GPU 사용 가능 여부 확인
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"\n✓ 사용 디바이스: {device}")

        # YOLOv8 모델 로드
        model_name = f'yolov8{model_size}.pt'
        print(f"✓ 모델 로드: {model_name}")
        self.model = YOLO(model_name)

        print(f"\n{'='*50}")
        print(f"학습 시작 - 32개 차량 부위 분류")
        print(f"{'='*50}\n")

        # 모델 학습
        results = self.model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=device,
            workers=4,
            patience=patience,
            save=True,
            project='car_parts_detection',
            name='exp',
            exist_ok=True,

            # 하이퍼파라미터
            lr0=lr0,
            lrf=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3.0,
            warmup_momentum=0.8,

            # Data augmentation
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=0.0,
            translate=0.1,
            scale=0.5,
            shear=0.0,
            perspective=0.0,
            flipud=0.0,
            fliplr=0.5,
            mosaic=1.0,
            mixup=0.0,
        )

        print(f"\n{'='*50}")
        print(f"학습 완료!")
        print(f"모델 저장 경로: {results.save_dir}")
        print(f"{'='*50}\n")

        return results

    def validate(self, model_path='car_parts_detection/exp/weights/best.pt'):
        """
        학습된 모델 검증

        Args:
            model_path: 검증할 모델 경로
        """
        print(f"\n{'='*50}")
        print(f"모델 검증 시작")
        print(f"{'='*50}\n")

        self.model = YOLO(model_path)
        self.class_names = self.model.names

        # 검증 데이터셋으로 평가
        yaml_path = self.create_yaml_config()
        metrics = self.model.val(data=yaml_path)

        print(f"\n{'='*50}")
        print(f"검증 결과:")
        print(f"mAP50: {metrics.box.map50:.4f}")
        print(f"mAP50-95: {metrics.box.map:.4f}")
        print(f"{'='*50}\n")

        return metrics

    def load_model(self, model_path='car_parts_detection/exp/weights/best.pt'):
        """학습된 모델 로드"""
        self.model = YOLO(model_path)
        self.class_names = self.model.names
        print(f"✓ 모델 로드 완료: {model_path}")

    def predict_image(self, image_path, conf_threshold=0.25, save_result=True):
        """
        이미지에서 차량 부위 감지

        Args:
            image_path: 입력 이미지 경로
            conf_threshold: 신뢰도 임계값
            save_result: 결과 이미지 저장 여부
        """
        if self.model is None:
            raise ValueError("모델이 로드되지 않았습니다. load_model()을 먼저 실행하세요.")

        print(f"\n{'='*50}")
        print(f"이미지 추론: {image_path}")
        print(f"{'='*50}\n")

        # 추론 실행
        results = self.model.predict(
            source=image_path,
            conf=conf_threshold,
            save=save_result,
            show_labels=True,
            show_conf=True,
            line_width=2
        )

        # 결과 출력
        for result in results:
            boxes = result.boxes
            print(f"감지된 부위 수: {len(boxes)}\n")

            for i, box in enumerate(boxes, 1):
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = self.class_names[cls_id]
                bbox = box.xyxy[0].cpu().numpy()

                print(f"{i}. {class_name}")
                print(f"   신뢰도: {conf:.2%}")
                print(f"   좌표: [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]\n")

        return results

    def predict_video(self, video_path, conf_threshold=0.25, save_result=True):
        """
        비디오에서 차량 부위 감지

        Args:
            video_path: 입력 비디오 경로
            conf_threshold: 신뢰도 임계값
            save_result: 결과 비디오 저장 여부
        """
        if self.model is None:
            raise ValueError("모델이 로드되지 않았습니다. load_model()을 먼저 실행하세요.")

        print(f"\n비디오 추론 시작: {video_path}")

        results = self.model.predict(
            source=video_path,
            conf=conf_threshold,
            save=save_result,
            stream=True
        )

        frame_count = 0
        for r in results:
            frame_count += 1
            print(f"프레임 {frame_count}: 감지된 객체 {len(r.boxes)}개")

        print(f"✓ 비디오 추론 완료: 총 {frame_count} 프레임")

    def predict_webcam(self, conf_threshold=0.25):
        """웹캠 실시간 감지"""
        if self.model is None:
            raise ValueError("모델이 로드되지 않았습니다. load_model()을 먼저 실행하세요.")

        print("\n웹캠 실시간 감지 시작 (종료: 'q' 키)")

        results = self.model.predict(
            source=0,
            conf=conf_threshold,
            show=True,
            stream=True
        )

        for r in results:
            pass

    def get_statistics(self, image_path, conf_threshold=0.25):
        """
        감지된 차량 부위 통계 반환

        Args:
            image_path: 입력 이미지 경로
            conf_threshold: 신뢰도 임계값

        Returns:
            dict: 각 부위별 감지 개수
        """
        if self.model is None:
            raise ValueError("모델이 로드되지 않았습니다. load_model()을 먼저 실행하세요.")

        results = self.model.predict(
            source=image_path,
            conf=conf_threshold,
            verbose=False
        )

        stats = {}
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls_id = int(box.cls[0])
                class_name = self.class_names[cls_id]
                stats[class_name] = stats.get(class_name, 0) + 1

        return stats

    def visualize(self, image_path, conf_threshold=0.25, save_path='result_custom.jpg'):
        """
        커스텀 시각화

        Args:
            image_path: 입력 이미지 경로
            conf_threshold: 신뢰도 임계값
            save_path: 결과 저장 경로
        """
        if self.model is None:
            raise ValueError("모델이 로드되지 않았습니다. load_model()을 먼저 실행하세요.")

        # 이미지 읽기
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")

        # 추론 실행
        results = self.model.predict(
            source=image_path,
            conf=conf_threshold,
            verbose=False
        )

        # 결과 그리기
        for result in results:
            boxes = result.boxes

            for box in boxes:
                # 바운딩 박스 좌표
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

                # 클래스 및 신뢰도
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = self.class_names[cls_id]

                # 색상 설정
                color = self.colors.get(class_name, (255, 255, 255))

                # 바운딩 박스 그리기
                cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

                # 레이블 그리기
                label = f'{class_name} {conf:.2f}'
                label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                cv2.rectangle(image, (x1, y1 - label_size[1] - 10),
                            (x1 + label_size[0], y1), color, -1)
                cv2.putText(image, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

        # 결과 저장
        cv2.imwrite(save_path, image)
        print(f"✓ 시각화 결과 저장: {save_path}")

        # 결과 표시
        cv2.imshow('Car Parts Detection', image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

        return image

    def export_model(self, model_path='car_parts_detection/exp/weights/best.pt',
                     format='onnx'):
        """
        모델을 다양한 포맷으로 내보내기

        Args:
            model_path: 내보낼 모델 경로
            format: 내보내기 포맷 (onnx, torchscript, engine 등)
        """
        model = YOLO(model_path)
        model.export(format=format)
        print(f"✓ 모델 내보내기 완료: {format} 포맷")


def main():
    """메인 실행 함수 - 순차적으로 학습 및 검증 실행"""

    print("""
    ╔═══════════════════════════════════════════════════════════╗
    ║       YOLOv8 차량 부위 분류 시스템 (32개 클래스)         ║
    ╚═══════════════════════════════════════════════════════════╝
    """)

    # CarPartsYOLO 인스턴스 생성
    detector = CarPartsYOLO()

    # ========================================
    # 1단계: 모델 학습
    # ========================================
    print("\n" + "="*60)
    print("1단계: 모델 학습 시작")
    print("="*60)

    # 학습 설정
    MODEL_SIZE = 'n'      # 모델 크기: n(nano), s(small), m(medium), l(large), x(xlarge)
    EPOCHS = 100          # 학습 에포크 수
    BATCH_SIZE = 16       # 배치 크기
    IMG_SIZE = 640        # 이미지 크기
    LEARNING_RATE = 0.01  # 초기 학습률

    print(f"""
학습 설정:
  - 모델 크기: YOLOv8{MODEL_SIZE}
  - 에포크: {EPOCHS}
  - 배치 크기: {BATCH_SIZE}
  - 이미지 크기: {IMG_SIZE}
  - 학습률: {LEARNING_RATE}
    """)

    # 학습 실행
    try:
        detector.train(
            model_size=MODEL_SIZE,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            lr0=LEARNING_RATE
        )
    except Exception as e:
        print(f"\n❌ 학습 중 오류 발생: {e}")
        return

    # ========================================
    # 2단계: 모델 검증
    # ========================================
    print("\n" + "="*60)
    print("2단계: 모델 검증 시작")
    print("="*60)

    MODEL_PATH = 'car_parts_detection/exp/weights/best.pt'

    try:
        metrics = detector.validate(MODEL_PATH)
    except Exception as e:
        print(f"\n❌ 검증 중 오류 발생: {e}")
        return

    # ========================================
    # 3단계: 테스트 이미지 추론 (옵션)
    # ========================================
    print("\n" + "="*60)
    print("3단계: 테스트 이미지 추론 (있는 경우)")
    print("="*60)

    # 테스트 이미지 경로 설정 (필요시 수정)
    TEST_IMAGE_PATH = r"/content/drive/MyDrive/balanced_dataset_split/test/images"
    CONF_THRESHOLD = 0.25

    # 테스트 이미지 디렉토리가 존재하는지 확인
    if os.path.exists(TEST_IMAGE_PATH) and os.path.isdir(TEST_IMAGE_PATH):
        # 첫 번째 이미지 파일 찾기
        image_files = [f for f in os.listdir(TEST_IMAGE_PATH)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

        if image_files:
            test_image = os.path.join(TEST_IMAGE_PATH, image_files[0])
            print(f"\n테스트 이미지: {test_image}")

            try:
                detector.load_model(MODEL_PATH)
                detector.predict_image(test_image, conf_threshold=CONF_THRESHOLD)

                # 통계 출력
                stats = detector.get_statistics(test_image, conf_threshold=CONF_THRESHOLD)
                if stats:
                    print("\n[감지된 부위 통계]")
                    for part, count in sorted(stats.items()):
                        print(f"  {part}: {count}개")
                else:
                    print("\n감지된 부위가 없습니다.")

            except Exception as e:
                print(f"\n❌ 추론 중 오류 발생: {e}")
        else:
            print(f"\n테스트 이미지를 찾을 수 없습니다: {TEST_IMAGE_PATH}")
    else:
        print(f"\n테스트 이미지 디렉토리가 존재하지 않습니다: {TEST_IMAGE_PATH}")

    # ========================================
    # 4단계: 모델 내보내기 (옵션)
    # ========================================
    print("\n" + "="*60)
    print("4단계: 모델 내보내기 (ONNX 포맷)")
    print("="*60)

    try:
        detector.export_model(MODEL_PATH, format='onnx')
    except Exception as e:
        print(f"\n❌ 내보내기 중 오류 발생: {e}")

    # ========================================
    # 완료
    # ========================================
    print("\n" + "="*60)
    print("✓ 모든 작업이 완료되었습니다!")
    print("="*60)
    print(f"""
결과 요약:
  - 학습된 모델: {MODEL_PATH}
  - ONNX 모델: car_parts_detection/exp/weights/best.onnx
  - 학습 로그: car_parts_detection/exp/
    """)


if __name__ == '__main__':
    main()


    ╔═══════════════════════════════════════════════════════════╗
    ║       YOLOv8 차량 부위 분류 시스템 (32개 클래스)         ║
    ╚═══════════════════════════════════════════════════════════╝
    

1단계: 모델 학습 시작

학습 설정:
  - 모델 크기: YOLOv8n
  - 에포크: 100
  - 배치 크기: 16
  - 이미지 크기: 640
  - 학습률: 0.01
    
✓ 학습 경로 확인: /content/drive/MyDrive/balanced_dataset_split/train/images
✓ 검증 경로 확인: /content/drive/MyDrive/balanced_dataset_split/val/images
✓ YAML 설정 파일 생성: car_parts.yaml

✓ 사용 디바이스: cuda
✓ 모델 로드: yolov8n.pt

학습 시작 - 32개 차량 부위 분류

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=car_parts.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, em